In [ ]:
# google colabの場合実行
# !uv pip install torch_geometric==2.8.1

In [ ]:
from IPython.display import clear_output

import numpy as np
import pandas as pd
import lightgbm as lgb
import torch
import torch.nn.functional as F
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from torch_geometric.datasets import Amazon
from torch_geometric.nn import GATConv, GCNConv

In [ ]:
seed = 42
y_col = "product_category"

## 1. データの読み込み

In [ ]:
dataset = Amazon(root="./data/Amazon", name="Computers")


In [ ]:
data = dataset[0]

## 2. 学習用のデータセットの作成

In [ ]:
num_nodes = data.num_nodes
num_features = dataset.num_features
num_classes = dataset.num_classes

In [ ]:
# インデックスで分割用
indices = np.arange(num_nodes)
y = data.y.numpy()

In [ ]:
# train:val:test=6:2:2にする

# test_val:test=8:2
train_val_idx, test_idx = train_test_split(
    indices, test_size=0.2, random_state=seed, stratify=y
)

# test_valの全体に対しての８割を1/4で分割するので、8*(1/4)=2となり残るが６になる
train_idx, val_idx = train_test_split(
    train_val_idx, test_size=0.25, random_state=seed, stratify=y[train_val_idx]
)

In [ ]:
# インデックスベースでマスクを作成
train_mask = torch.zeros(num_nodes, dtype=torch.bool)
val_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask = torch.zeros(num_nodes, dtype=torch.bool)

train_mask[train_idx] = True
val_mask[val_idx] = True
test_mask[test_idx] = True

# tensor形式のdataにマスクブール値を追加
data.train_mask = train_mask
data.val_mask = val_mask
data.test_mask = test_mask

In [ ]:
# lightgbm用にdataframe形式のデータセットを作成

X_df = pd.DataFrame(data.x.numpy())
y_df = pd.Series(data.y.numpy(), name=y_col)

X_train_df, y_train_df = X_df.iloc[train_idx], y_df.iloc[train_idx]
X_val_df, y_val_df = X_df.iloc[val_idx], y_df.iloc[val_idx]
X_test_df, y_test_df = X_df.iloc[test_idx], y_df.iloc[test_idx]



## 3. 学習および評価

### 3.1 lightgbm

In [ ]:
model_lgb = lgb.LGBMClassifier(
    objective="multiclass",
    # class_weight="balanced", #　不均衡データだけど、GNN系で不均衡対策を入れてないので、条件を合わせる。
    random_state=seed
)

In [ ]:
model_lgb.fit(
    X_train_df, y_train_df,
    eval_set=[(X_val_df, y_val_df)],
    callbacks=[lgb.early_stopping(stopping_rounds=50)]
)

In [ ]:
pred_lgb = model_lgb.predict(X_test_df)

In [ ]:
dict_report_lgb = classification_report(y_test_df, pred_lgb, output_dict=True,zero_division=0)

In [ ]:
pd.DataFrame(dict_report_lgb)

### 3.2 GNN

In [ ]:
def set_seed_for_gnn():

  np.random.seed(seed)

  torch.manual_seed(seed)
  if torch.cuda.is_available():
      torch.cuda.manual_seed_all(seed)

#### 3.2.1 モデル設定

In [ ]:
# GCNのモデルを作成
class GCN(torch.nn.Module):

    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    # 順伝播
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)  #畳み込みそう
        x = F.relu(x) #　活性化関数
        x = F.dropout(x, p=0.5, training=self.training) # dropout層
        x = self.conv2(x, edge_index)
        return x


In [ ]:
# GATのモデル作成

class GAT(torch.nn.Module):

    def __init__(self, in_channels, hidden_channels, out_channels, heads=8):
        super().__init__()
        # 出力次元が hidden_channels * heads になるため、マルチヘッドのアテンションを考慮
        # 最終層はクラス数(out_channels)に合わせるため、heads=1、concat=Falseにする
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads)
        self.conv2 = GATConv(
            hidden_channels * heads, out_channels, heads=1, concat=False
        )

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x

#### 3.2.2　モデルの学習と評価

In [ ]:
def train_node_classifier(model, data, epochs=200, lr=1e-2, weight_decay=5e-4, early_stopping_num=50):
    set_seed_for_gnn()
    # cudaならcuda。それ以外ならcpuを設定
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # モデルとデータをデバイスに移動
    model = model.to(device)
    data = data.to(device)

    # 最適化関数
    optimizer = torch.optim.Adam(
        model.parameters(), lr=lr, weight_decay=weight_decay
    )
    criterion = torch.nn.CrossEntropyLoss()

    best_val_loss = float("inf")
    best_state = None
    i = 0 #early stoppingの簡易版のカウンター

    for epoch in range(epochs):

        # trainフェーズ
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = criterion(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        # evalフェーズ(モデルを評価モード＋勾配を記録しない)

        model.eval()
        with torch.no_grad():
            logits = model(data.x, data.edge_index)
            val_loss = criterion(logits[data.val_mask],data.y[data.val_mask],).item()
            # preds = logits.argmax(dim=-1)

            # 各マスクでの精度(Accuracy)計算
            # train_acc = (
            #     (preds[data.train_mask] == data.y[data.train_mask])
            #     .float()
            #     .mean()
            #     .item()
            # )

        clear_output(wait=True)
        print(
            f"Epoch {epoch + 1}/{epochs} "
            f"- Train Loss: {loss.item():.4f} "
            f"- Val Loss: {val_loss:.4f}"
        )

        # 簡易的なearly stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            i = 0
        else:
            i += 1
        if i >= early_stopping_num:
            print("Early stopping")
            break

    # テストデータでの精度を計算

    # val_lossが一番良かったモデルに戻す
    if best_state is not None:
      model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
      logits = model(data.x, data.edge_index)
      preds = logits.argmax(dim=-1)

      y_true = data.y[data.test_mask].cpu().numpy()
      y_pred = preds[data.test_mask].cpu().numpy()

    dict_report = classification_report(
        y_true, y_pred,
        output_dict=True,
        zero_division=0
        )

    return pd.DataFrame(dict_report)

##### 3.2.2.1 GCN

In [ ]:
set_seed_for_gnn()

model_gcn = GCN(
      in_channels=num_features,
      hidden_channels=16,
      out_channels=num_classes,
  )

In [ ]:
epochs =500

gcn_report = train_node_classifier(
    model_gcn,
    data,
    epochs=epochs,
    lr=1e-2,
    weight_decay=5e-4,
    early_stopping_num=50
)


In [ ]:
gcn_report

##### 3.2.2.1 GAT

In [ ]:
set_seed_for_gnn()

model_gat = GAT(
    in_channels=num_features,
    hidden_channels=8,
    out_channels=num_classes,
    heads=8,
)

# マルチヘッド(8)、hidden_channnels８なので、中間層の次元は 8 * 8 = 64

In [ ]:
epochs =500

gat_report = train_node_classifier(
    model_gat,
    data,
    epochs=epochs,
    lr=1e-2,
    weight_decay=5e-4,
    early_stopping_num=50
)


In [ ]:
gat_report